In [2]:
import pandas as pd
import holidays


df = pd.read_csv(
    "dataclean_prm_30000250086126.csv",
    sep=",",
    decimal="."
)

df.head()

,prm,ville,puissance,datetime,id_site,code_postal,temperature,humidite,precipitation,vitesse_vent,direction_vent,couverture_nuages
0,30000250086126,Juvigny les Vallées,35000.0,2024-05-04 15:55:00,9.0,50520,14.7,59.0,0.1,14.5,145.0,100.0
1,30000250086126,Juvigny les Vallées,35000.0,2024-05-18 13:00:00,9.0,50520,18.0,61.0,0.1,5.4,352.0,100.0
2,30000250086126,Juvigny les Vallées,35000.0,2024-05-18 13:05:00,9.0,50520,18.0,61.0,0.1,5.4,352.0,100.0
3,30000250086126,Juvigny les Vallées,35000.0,2024-05-18 13:25:00,9.0,50520,18.0,61.0,0.1,5.4,352.0,100.0
4,30000250086126,Juvigny les Vallées,35000.0,2024-05-18 13:45:00,9.0,50520,18.0,61.0,0.1,5.4,352.0,100.0


In [3]:
import pandas as pd
import holidays

# -------------------------------
# 1️⃣ Préparer les infos temporelles
# -------------------------------

# Convertir en datetime
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

# Extraire la date
df["date"] = df["datetime"].dt.date
df["date"] = pd.to_datetime(df["date"])

# Extraire l'heure
df["heure"] = df["datetime"].dt.hour

# Jour de la semaine (0 = lundi, 6 = dimanche)
df['jour_semaine'] = df["datetime"].dt.dayofweek

# Mois de l'année
df['mois'] = df["datetime"].dt.month

# Weekend
df['weekend'] = (df['jour_semaine'] >= 5).astype(int)

# Jour férié
jours_feries_fr = holidays.France()
df["jour_ferie"] = df["date"].dt.normalize().map(lambda d: d.date() in jours_feries_fr).astype(int)

# -------------------------------
# 2️⃣ Agrégations par heure
# -------------------------------

# Ici, on peut grouper par PRM + date + heure
df_puissance_heure = (
    df.groupby(["prm", "date", "heure"])
    .agg(
        puissance_moy_heure=("puissance", "mean"),
        puissance_sum_heure=("puissance", "sum")
    )
    .reset_index()
)

# -------------------------------
# 3️⃣ Fusionner avec le dataframe original
# -------------------------------

df = df.merge(
    df_puissance_heure,
    on=["prm", "date", "heure"],
    how="left"
)

# -------------------------------
# 4️⃣ Nettoyage final
# -------------------------------

df = df.drop(columns=["puissance"])  # On peut garder 'datetime' si besoin
df = df.drop_duplicates(subset=["prm", "date", "heure"])

print(f"[INFO] Shape finale (1 ligne par PRM et par heure) : {df.shape}")

[INFO] Shape finale (1 ligne par PRM et par heure) : (17543, 19)


In [4]:
df.head(10)

,prm,ville,datetime,id_site,code_postal,temperature,humidite,precipitation,vitesse_vent,direction_vent,couverture_nuages,date,heure,jour_semaine,mois,weekend,jour_ferie,puissance_moy_heure,puissance_sum_heure
0,30000250086126,Juvigny les Vallées,2024-05-04 15:55:00,9.0,50520,14.7,59.0,0.1,14.5,145.0,100.0,2024-05-04,15,5,5,1,0,49750.000000,597000.0
1,30000250086126,Juvigny les Vallées,2024-05-18 13:00:00,9.0,50520,18.0,61.0,0.1,5.4,352.0,100.0,2024-05-18,13,5,5,1,0,31666.666667,380000.0
5,30000250086126,Juvigny les Vallées,2025-04-12 16:00:00,9.0,50520,18.1,64.0,0.1,18.4,234.0,100.0,2025-04-12,16,5,4,1,0,48166.666667,578000.0
6,30000250086126,Juvigny les Vallées,2025-05-10 20:05:00,9.0,50520,18.0,54.0,0.1,11.8,101.0,100.0,2025-05-10,20,5,5,1,0,35166.666667,422000.0
12,30000250086126,Juvigny les Vallées,2025-05-11 14:25:00,9.0,50520,19.2,55.0,0.1,21.6,163.0,100.0,2025-05-11,14,6,5,1,0,33166.666667,398000.0
17,30000250086126,Juvigny les Vallées,2025-05-09 14:15:00,9.0,50520,17.6,49.0,0.1,19.8,74.0,100.0,2025-05-09,14,4,5,0,0,49083.333333,589000.0
39,30000250086126,Juvigny les Vallées,2025-05-06 13:00:00,9.0,50520,12.9,51.0,0.1,22.4,33.0,100.0,2025-05-06,13,1,5,0,0,471166.666667,5654000.0
51,30000250086126,Juvigny les Vallées,2024-08-07 14:00:00,9.0,50520,19.8,58.0,0.1,13.9,260.0,100.0,2024-08-07,14,2,8,0,0,356250.000000,4275000.0
63,30000250086126,Juvigny les Vallées,2024-05-19 15:00:00,9.0,50520,19.5,57.0,0.1,4.2,59.0,100.0,2024-05-19,15,6,5,1,0,61833.333333,742000.0
75,30000250086126,Juvigny les Vallées,2024-05-15 13:00:00,9.0,50520,16.5,60.0,0.1,19.2,163.0,100.0,2024-05-15,13,2,5,0,0,420083.333333,5041000.0


In [19]:
from pathlib import Path


df.to_csv("dataFE_prm_30000250086126.csv",
    index=False
)